In [1]:
import torch
import numpy as np
from pathlib import Path
from cleanfid import fid
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import pandas as pd
from tqdm import tqdm
import json
import matplotlib.pyplot as plt

In [4]:
from datasets import load_dataset
import pandas as pd

# LAION Aesthetics V2 (most commonly cited)
ds = load_dataset("laion/aesthetics_v2_4.5", split="train", streaming=True)

# Take first 5000 entries
subset = []
for i, item in enumerate(ds):
    if i >= 5000:
        break
    subset.append({
        "url": item["URL"],
        "text": item["TEXT"],       # ← this is your text prompt
    })

df = pd.DataFrame(subset)
df.to_csv("laion_5k_prompts.csv", index=False)
print(f"Saved {len(df)} rows")
print(df.head())

Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

Saved 5000 rows
                                                 url  \
0  https://i.ebayimg.com/images/g/iiIAAOSw0j9ZU6t...   
1  https://www.vector-eps.com/wp-content/gallery/...   
2   https://bikes.rim.de/bikid-69184-800-800-0-0.jpg   
3  https://images.complex.com/complex/images/c_li...   
4  https://lmr.net.au/wp-content/uploads/2013/12/...   

                                                text  
0  2PCS  69*71mm Green Plastic Bobbin Wire Coil F...  
1  License you can use spring labels vector desig...  
2                               Puky - Racer Angebot  
3       The 25 Best Actresses Who Never Won An Oscar  
4  roofing gold coast, colorbond roofing gold coa...  


In [ ]:
import requests
from pathlib import Path
import pandas as pd
from PIL import Image
from io import BytesIO

real_dir = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/figures/laion_5k_real")
real_dir.mkdir(exist_ok=True)

df = pd.read_csv("laion_5k_prompts.csv")
failed = 0

for idx, row in df.iterrows():
    img_path = real_dir / f"{idx:05d}.png"
    if img_path.exists():
        continue  # resume if interrupted
    try:
        r = requests.get(row["url"], timeout=5)
        r.raise_for_status()
        # Convert to PNG via PIL to avoid saving corrupt/non-image bytes
        img = Image.open(BytesIO(r.content)).convert("RGB")
        img.save(img_path)
    except Exception as e:
        failed += 1

print(f"Done. {len(df) - failed}/{len(df)} images saved, {failed} failed.")

/n/home00/zoewu/.local/lib/python3.12/site-packages/PIL/Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [ ]:

# ── Config ────────────────────────────────────────────────────────────────────
REAL_DIR = "/figures/laion_5k_real"
TSR_DIR = Path("/figures/laion_5k_generated")
PT_TSR_DIR = Path("/figures/laion_5k_generated_pt")
PROMPTS_FILE = "laion_5k_prompts.csv"
K_VALUES = [1.0, 0.98]

df = pd.read_csv(PROMPTS_FILE)
prompts = df["TEXT"].tolist()[:100]

# ── CLIP setup ────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def compute_clip(gen_dir):
	scores = []
	for idx, prompt in enumerate(tqdm(prompts, desc=f"CLIP {gen_dir.name}")):
		img_path = gen_dir / f"{idx:05d}.png"
		if not img_path.exists():
			continue
		image = Image.open(img_path).convert("RGB")
		inputs = clip_processor(text=[prompt], images=image, return_tensors="pt", padding=True).to(device)
		with torch.no_grad():
			score = clip_model(**inputs).logits_per_image[0, 0].item()
		scores.append(score)
	return np.mean(scores)

def compute_fid_score(gen_dir):
	return fid.compute_fid(REAL_DIR, str(gen_dir))

# ── Compute for both TSR and PT-TSR ──────────────────────────────────────────
tsr_results    = {}  # {k: (fid, clip)}
pt_tsr_results = {}

for tsr_k in K_VALUES:
	k_str = f"k{tsr_k:.3f}".replace(".", "p")

	print(f"{k_str}")

	# TSR
	tsr_gen_dir = TSR_DIR / k_str
	tsr_fid  = compute_fid_score(tsr_gen_dir)
	tsr_clip = compute_clip(tsr_gen_dir)
	tsr_results[tsr_k] = (tsr_fid, tsr_clip)
	print(f"[TSR]    k={tsr_k:.3f}  FID={tsr_fid:.4f}  CLIP={tsr_clip:.4f}")

	# PT-TSR
	pt_gen_dir = PT_TSR_DIR / k_str
	pt_fid  = compute_fid_score(pt_gen_dir)
	pt_clip = compute_clip(pt_gen_dir)
	pt_tsr_results[tsr_k] = (pt_fid, pt_clip)
	print(f"[PT-TSR] k={tsr_k:.3f}  FID={pt_fid:.4f}  CLIP={pt_clip:.4f}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n── FID (lower is better) ──")
for k in K_VALUES:
	print(f"  k={k:.3f}  TSR={tsr_results[k][0]:.4f}  PT-TSR={pt_tsr_results[k][0]:.4f}")

print("\n── CLIP (higher is better) ──")
for k in K_VALUES:
	print(f"  k={k:.3f}  TSR={tsr_results[k][1]:.4f}  PT-TSR={pt_tsr_results[k][1]:.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

# TSR line
tsr_k_vals = sorted(tsr_results.keys(), reverse=True)
tsr_clip   = [tsr_results[k][1] for k in tsr_k_vals]
tsr_fid    = [tsr_results[k][0] for k in tsr_k_vals]
ax.plot(tsr_clip, tsr_fid, color="gold", marker="o", linewidth=2, label="TSR, CFG=7.5, σ=3.0")
for k in tsr_k_vals:
	f, c = tsr_results[k]
	ax.annotate(f"k={k}", (c, f), textcoords="offset points", xytext=(6, 0), fontsize=8, color="goldenrod")

# PT-TSR line
pt_k_vals = sorted(pt_tsr_results.keys(), reverse=True)
pt_clip   = [pt_tsr_results[k][1] for k in pt_k_vals]
pt_fid    = [pt_tsr_results[k][0] for k in pt_k_vals]
ax.plot(pt_clip, pt_fid, color="orangered", marker="o", linewidth=2, label="PT-TSR, CFG=7.5, σ=3.0")
for k in pt_k_vals:
	f, c = pt_tsr_results[k]
	ax.annotate(f"k={k}", (c, f), textcoords="offset points", xytext=(6, 0), fontsize=8, color="orangered")

ax.set_xlabel("CLIP", fontsize=12)
ax.set_ylabel("FID", fontsize=12)
ax.set_title("FID vs CLIP comparison", fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fid_vs_clip.png", dpi=150)
plt.show()
print("Saved fid_vs_clip.png")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/n/home00/zoewu/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


k1p000
compute FID between two folders
Found 0 images in the folder /path/to/laion_5k_real_images


FID laion_5k_real_images : : 0it [00:00, ?it/s]


ValueError: need at least one array to concatenate